# Comparison — v2 (aggregation) vs v3 (md0 chain)

Builds the intra-US MRIOT both ways on the **same** 2017 data and bilateral flows, then
reproduces the identity comparison of the original `build_IOT` notebook (row + column
identities, ΣZ/ΣF, residuals), plus a decomposition of why the column identity differs.

- **margins excluded (ref)** — old v1.1 baseline: local domestic margins `dm0` left out.
- **v2 aggregation** — margins folded into goods (`reconstruct_bilateral_2` combined flow,
  local `xd0 = dd0 + dm0`, single use-share set). Same construction as `v2_construction`.
- **v3 md0 chain** — margins routed separately to the buyers that carry them. Same as
  `v3_construction`.

Both close the **row** (supply) identity; they differ on the **column** (cost) identity.

## Setup

# Librairies

#### Installations

In [ ]:
from paths import ROOT
import sys
!{sys.executable} -m pip install gdx2py

In [ ]:
import sys
!{sys.executable} -m pip install gamspy-base

#### Imports

In [ ]:
from gdx2py import GdxFile
import gamspy_base
import os
import pandas as pd
from gdx2py.gams import GAMSParameter, GAMSSet
import numpy as np
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from itertools import product

In [ ]:
gamspybase_directory = gamspy_base.directory
print(gamspybase_directory)

In [ ]:
path_windc_gdx = str(ROOT / "data/raw/GTAPWiNDC/data/core/WiNDCdatabase.gdx")

In [ ]:
gdx = GdxFile(path_windc_gdx, gams_dir=gamspybase_directory)
print(list(gdx))

In [ ]:
# ── 1. Load all parameters ───────────────────────────────────────────────────
params = {}
for name, obj in gdx:
    if isinstance(obj, GAMSParameter):
        s = obj.to_pandas()
        if s is not None and len(s) > 0:
            df = s.reset_index()
            df.columns = list(df.columns[:-1]) + ['value']
            params[name] = df
# params contains all years and regions. We will filter it later when we need to build the IOT for a specific year and region.

In [ ]:
# Regions: union of all states present in xn0_ or nd0_
all_xn0 = set(params['xn0_']['r'].unique())
all_nd0 = set(params['nd0_']['r'].unique())
regions = sorted(all_xn0 | all_nd0)
n = len(regions)
print(n)

In [ ]:
# Economic (GDP-weighted) centroids -- the delivered reference points of the gravity
# distance matrix. Built by 02_economic_centroids.ipynb (BEA CAGDP2 county GDP + the
# 2020 Census county population centroids). Loaded here as {abbr: (lat, lon)}.
_cen = pd.read_csv(ROOT / "data/interim/economic_centroids.csv")
COORDS = {r.abbr: (r.lat, r.lon) for r in _cen.itertuples(index=False)}


In [ ]:
# distance function
def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi, dlam = np.radians(lat2 - lat1), np.radians(lon2 - lon1)
    a = np.sin(dphi/2)**2 + np.cos(phi1)*np.cos(phi2)*np.sin(dlam/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

In [ ]:
# Distance matrix between all pairs of regions, great-circle (haversine) between the
# GDP-weighted ECONOMIC CENTROIDS of each region (data/interim/economic_centroids.csv,
# built in 02_economic_centroids.ipynb). These are the delivered reference points; the
# earlier capital-based prior and the alternative (geometric, population-weighted)
# centroids are compared in analysis/distance_variants.py of the source project.
D= pd.DataFrame(index=regions, columns=regions, dtype=float)
for i, j in product(regions, regions):
    D.loc[i, j] = np.nan if i == j else haversine(*COORDS[i], *COORDS[j])

D_np = D.values.copy()

missing = [r for r in regions if r not in COORDS]
if missing:
    print(f"Warning: regions without coordinates: {missing}")
print(f"{len(regions)} regions | distance range: "
      f"{D_np[~np.isnan(D_np)].min():.0f}-{D_np[~np.isnan(D_np)].max():.0f} km")


In [ ]:
# Example: load use matrix for New York 2017 as a numpy array (goods × sectors)
def load_matrix(param_name, year, regions, sectors):
    df = params[param_name]
    dim = 'g' if 'g' in df.columns else 's'
    return (df[df['yr'] == year]
            .groupby(['r', dim])['value'].sum()
            .unstack(dim)
            .reindex(index=regions, columns=sectors, fill_value=0.0)
            .fillna(0.0)          
            .values)

In [ ]:
def load_year_data(year, regions, sectors, names_params):
    """Load all IO matrices for a given year. Returns a dict of arrays."""
    n, S = len(regions), len(sectors)

    names = names_params
    mats = {name: load_matrix(name, year, regions, sectors) for name in names}

    absorption = mats['dd0_'] + mats['nd0_'] + mats['m0_']
    absorption_safe = np.where(absorption < 1e-10, 1.0, absorption)

    id0_tensor = (params['id0_'][params['id0_']['yr'] == year]
                  .groupby(['r', 'g', 's'])['value'].sum()
                  .unstack('s')
                  .reindex(pd.MultiIndex.from_product([regions, sectors], names=['r', 'g']),
                           fill_value=0.0)
                  .reindex(columns=sectors, fill_value=0.0)
                  .fillna(0.0)
                  .values
                  .reshape(n, S, S))

    return {**mats, 'absorption': absorption, 'absorption_safe': absorption_safe,
            'id0': id0_tensor}

In [ ]:
EXCLUDED = {'fen', 'sle'}
sectors = sorted(s for s in params['xn0_']['g'].unique() if s not in EXCLUDED)

In [ ]:
# Index maps and dimensions
region_to_idx = {r: i for i, r in enumerate(regions)}
sector_to_idx = {s: i for i, s in enumerate(sectors)}
n, S = len(regions), len(sectors)
print(f'{n} regions x {S} sectors')

## Functions (both methods)

In [ ]:
def ras_robust(seed, X, M, max_iter=2000, tol=1e-8):
    """RAS with convergence tracking. RAS lets us preserve the table's initial aggregate structure at the start.
    Indeed the approximation of the formula T = X.M.D^gamma distorts the matrix and does not guarantee that the resulting table
    is consistent with the aggregate flows observed at the start,
    i.e. that the total of what leaves as good g from state i toward states j equals the exports of state i for good g
    toward the NP in the initial table (row agreement);
    and that the total of what enters as good g into state j from states i equals the imports of state j for good g
    from the NP in the initial table (column agreement).

    The RAS algorithm proportionally scales up a whole row and a whole column at each iteration,
    until the row and column totals are close enough to the target totals (X and M).

    Args:
    seed: starting matrix (nxn)
    X: vector of row totals (n,)
    M: vector of column totals (n,)
    max_iter: maximum number of iterations
    tol: convergence tolerance

    Returns:
        T: adjusted matrix
        converged: boolean indicating whether convergence was reached for each sector
        final_err: final error (max of the deviations from the totals)
        iters: number of iterations performed
        initial_err: initial error (before adjustment)
    """
    T = seed.copy().astype(float) #starting nxn matrix created from the gravity seed with the chosen gamma
    r = X.values if hasattr(X, 'values') else X
    c = M.values if hasattr(M, 'values') else M
    initial_err = max(np.abs(T.sum(axis=1) - r).max(),
                      np.abs(T.sum(axis=0) - c).max())
    for it in range(max_iter):
        rs = T.sum(axis=1); rs[rs == 0] = 1
        T *= (r / rs)[:, None]
        cs = T.sum(axis=0); cs[cs == 0] = 1
        T *= (c / cs)[None, :]
        err = max(np.abs(T.sum(axis=1) - r).max(),
                  np.abs(T.sum(axis=0) - c).max())
        if err < tol:
            return T, True, err, it + 1, initial_err
    return T, False, err, max_iter, initial_err

In [ ]:
def reconstruct_bilateral_2(xn0_mat, nd0_mat, nm0_mat, sectors, sector_to_idx, regions,
                          D_np, gamma=1.0, imbalance_skip=0.50):
    """
    Reconstruct bilateral trade matrices T(region i→ region j, good g) via gravity model + RAS.
    Import target = nd0 + nm0 (direct absorption + margin absorption).
    Export target = xn0 (exports to national pool).
    Returns :
    T_all (dict sector→n×n array, n=regions) 
    df_log (convergence log).
    """
    n = len(regions)

    with np.errstate(divide='ignore', invalid='ignore'):
        friction = np.where(np.isnan(D_np), 0.0, D_np ** (-gamma))

    T_all = {}
    log_ras = []

    for g in sectors:
        g_i = sector_to_idx[g]
        X_g = pd.Series(xn0_mat[:, g_i] , index=regions)
        M_g = pd.Series(nd0_mat[:, g_i] + nm0_mat[:, g_i], index=regions) # for margin sectors, a part of their production is absorbed as margin (nm0) rather than direct use (nd0), but both contribute to the "import" side that RAS should match
        total_X = X_g.sum()
        total_M = M_g.sum()

        imbalance = abs(total_X - total_M) / total_X
        if total_M < 1e-10 or imbalance > imbalance_skip:
            T_all[g] = np.zeros((n, n))
            log_ras.append({'sector': g, 'status': 'skipped_imbalance',
                            'err': imbalance, 'iters': 0})
            continue

        #if imbalance > 1e-6:
            #M_g = M_g * (total_X / total_M)

        seed = np.outer(X_g.values, M_g.values) * friction
        seed += 1e-8 * np.outer(X_g.values / total_X, M_g.values / M_g.sum())
        np.fill_diagonal(seed, 0.0)

        T_g, converged, err, iters, initial_err = ras_robust(seed, X_g, M_g)
        
        tol_soft = 1e-6  # relaxed tolerance for borderline cases
        if converged:
            status = 'ok'
        elif err < tol_soft:
            status = 'ok_soft'   # converged to relaxed tolerance
        else:
            status = 'FAILED'

        T_all[g] = T_g
        log_ras.append({'sector': g, 'status': status, 'initial_err': initial_err,
                        'err': err, 'iters': iters, 'seed': 'gravity'})
        

    return T_all, pd.DataFrame(log_ras)


In [ ]:
def reconstruct_bilateral_3(xn0_mat, nd0_mat, nm0_mat, sectors, sector_to_idx, regions,
                            D_np, gamma_trade=1.0, gamma_margin=1.0,
                            imbalance_skip=1e-4):
    """
    Reconstruct bilateral matrices separately for trade links (nd0) and margin
    links (nm0), each with its own spatial distribution, via gravity + RAS.

    Both flows draw from the same export pool xn0 (row/origin marginal) but have
    distinct absorption targets (column/destination marginals): nd0 for direct
    trade, sum_m nm0 for margin absorption. Per commodity, the export pool is
    split in proportion to the national total absorbed through each channel, so
    that each layer carries equal row and column totals:

        sum_r X_trade(r)  == sum_r nd0(r)        (per commodity)
        sum_r X_margin(r) == sum_r nm0(r)        (per commodity)

    That equality is the FEASIBILITY condition of the RAS, not a property of the
    gravity seed: the raw product X.M.d^-gamma reproduces neither marginal. It is
    ras_robust that restores them, so that after fitting

        row_sums(T_trade) + row_sums(T_margin) = xn0        (per region)
        col_sums(T_trade)                      = nd0        (per region)
        col_sums(T_margin)                     = sum_m nm0  (per region)

    to the fitting tolerance. The row identity holds only when total_M == total_X,
    since the split rescales the pool by total_M / total_X; the WiNDC identity
    sum_r xn0 = sum_r (nd0 + sum_m nm0) makes that exact per commodity, and the
    imbalance guard below is what catches a source where it is not.

    Note that RAS absorbs any monotone rescaling of the row and column masses into
    its own diagonal scalings, so exponents on X and M would leave T unchanged:
    the only parameter of the seed that survives the fit is gamma.

    Returns:
        T_trade_all  (dict sector -> n x n array)
        T_margin_all (dict sector -> n x n array)
        df_log       (convergence log).
    """
    n = len(regions)

    with np.errstate(divide='ignore', invalid='ignore'):
        friction_trade  = np.where(np.isnan(D_np), 0.0, D_np ** (-gamma_trade))
        friction_margin = np.where(np.isnan(D_np), 0.0, D_np ** (-gamma_margin))

    def _ras_one_flow(X_row, M_col, friction):
        """Run a single gravity+RAS reconstruction; X_row/M_col are pd.Series."""
        tX, tM = X_row.sum(), M_col.sum()
        if tX < 1e-10 or tM < 1e-10:           # flow absent -> empty matrix
            return np.zeros((n, n)), True, 0.0, 0, 0.0
        # No epsilon floor is added to the seed. It would be redundant: friction is
        # d^-gamma with d finite and strictly positive off the diagonal, so the seed
        # has no structural zero for RAS to trip on, and a zero row/column of X or M
        # carries a zero target anyway. Measured on 2017, adding the former
        # 1e-8 * outer(X/tX, M/tM) term left the iteration count identical on all 79
        # (commodity, layer) fits and only perturbed the result, because
        # outer(X,M)*f + 1e-8*outer(X/tX,M/tM) == outer(X,M) * (f + 1e-8/(tX*tM)),
        # i.e. it was a scale-DEPENDENT additive perturbation of the friction: 5e-9 of
        # the smallest friction on a median commodity but 1.1% of it on pipeline
        # transport, the smallest pool, shifting that layer by 5.1e-4 in relative L1.
        seed = np.outer(X_row.values, M_col.values) * friction
        np.fill_diagonal(seed, 0.0)
        return ras_robust(seed, X_row, M_col)

    def _status(converged, err, tol_soft=1e-6):
        if converged:        return 'ok'
        if err < tol_soft:   return 'ok_soft'
        return 'FAILED'

    T_trade_all, T_margin_all = {}, {}
    log_ras = []

    for g in sectors:
        g_i = sector_to_idx[g]
        X_g  = pd.Series(xn0_mat[:, g_i], index=regions)
        nd_g = pd.Series(nd0_mat[:, g_i], index=regions)
        nm_g = pd.Series(nm0_mat[:, g_i], index=regions)

        total_X  = X_g.sum()
        total_nd = nd_g.sum()
        total_nm = nm_g.sum()
        total_M  = total_nd + total_nm

        # Two distinct cases, which the previous single test conflated.
        # (a) The commodity has no national pool activity at all: nothing to
        #     reconstruct, an empty layer pair is the correct answer.
        if total_X < 1e-10 or total_M < 1e-10:
            T_trade_all[g]  = np.zeros((n, n))
            T_margin_all[g] = np.zeros((n, n))
            log_ras.append({'sector': g, 'status': 'no_pool_activity',
                            'err': 0.0, 'iters': 0})
            continue

        # (b) The marginals disagree. WiNDC closes the pool per commodity,
        #     sum_r xn0 == sum_r (nd0 + sum_m nm0), so any disagreement beyond float
        #     accumulation means the inputs are misaligned (wrong year, wrong margin
        #     axis, a source vintage that broke the identity) and the fit below would
        #     silently return a rescaled table. Measured over 1997-2023 x 71
        #     commodities the worst residual is 2.5e-7 (pipeline transport, 2005) and
        #     nothing exceeds 1e-6, whereas dropping nm0 from the marginal - the
        #     documented failure mode - puts 8 commodities above 0.5. The threshold
        #     sits between the two, and this is now an error rather than a silent
        #     zero-fill: a dropped commodity used to disappear from the table with no
        #     trace outside the log.
        imbalance = abs(total_X - total_M) / total_X
        if imbalance > imbalance_skip:
            raise ValueError(
                f"national pool marginals disagree for commodity {g!r}: "
                f"sum xn0 = {total_X:.6g}, sum (nd0 + nm0) = {total_M:.6g}, "
                f"relative imbalance {imbalance:.3e} > {imbalance_skip:.0e}. "
                f"The WiNDC pool identity should close this to ~1e-7; check that "
                f"nm0 is summed over margin types and that all inputs are the same "
                f"year.")

        # Split each region's export pool between trade and margin proportionally
        # to their (rescaled) national totals -> each flow is self-balanced.
        X_trade  = X_g * (total_nd / total_X)
        X_margin = X_g * (total_nm / total_X)

        T_trade,  c_t, err_t, it_t, _ = _ras_one_flow(X_trade,  nd_g, friction_trade)
        T_margin, c_m, err_m, it_m, _ = _ras_one_flow(X_margin, nm_g, friction_margin)

        T_trade_all[g]  = T_trade
        T_margin_all[g] = T_margin

        log_ras.append({'sector': g,
                        'status_trade':  _status(c_t, err_t),
                        'status_margin': _status(c_m, err_m),
                        'err_trade': err_t, 'err_margin': err_m,
                        'iters_trade': it_t, 'iters_margin': it_m,
                        'nd_total': total_nd, 'nm_total': total_nm,
                        'seed': 'gravity'})

    return T_trade_all, T_margin_all, pd.DataFrame(log_ras)


In [ ]:
def compute_use_shares_2(id0_df, cd0_mat, i0_mat, g0_mat, a0_mat=None):
    """
    Compute use shares (intermediate + final demand) at PURCHASER prices.

    For each (region r, good g):
        use_share_interm[r, g, s] = id0[r, g, s] / total_demand[r, g]
        use_share_C/I/G[r, g]     = cd0/i0/g0[r, g] / total_demand[r, g]
    with total_demand = id0.sum_s + cd0 + i0 + g0  (purchaser-price absorption).
    By construction the shares sum to 1 over {sectors s} + {C, I, G}.


    Fallback for margin goods where all demand is zero (id0=cd0=i0=g0=0):
    use_share_interm is set proportional to each sector's total intermediate
    purchases so that nm0/dm0 flows are not lost in build_Z.

    Arguments
    ---------
    id0_df  : (n, S, S) intermediary demand by (r,g,s)
    cd0_mat : (n, S)    household consumption demand by (r,g)
    i0_mat  : (n, S)    investment demand by (r,g)
    g0_mat  : (n, S)    government consumption demand by (r,g)
    a0_mat  : ignored (deprecated -- see note above)
    """
    # Purchaser-price absorption of good g in region r:
    # total intermediate demand (sum over buying sectors s) + final demand.
    total_demand = id0_df.sum(axis=2) + cd0_mat + i0_mat + g0_mat  # (n, S)
    safe = np.where(total_demand < 1e-10, 1.0, total_demand)       # (n, S)

    use_share_interm = id0_df  / safe[:, :, None]   # (n, S, S)
    use_share_C      = cd0_mat / safe               # (n, S)
    use_share_I      = i0_mat  / safe               # (n, S)
    use_share_G      = g0_mat  / safe               # (n, S)

    # --------------------------------------------------------------------------
    # FALLBACK FOR MARGIN GOODS WITH NO RECORDED DEMAND (id0=cd0=i0=g0=0).
    #
    # Margin goods (trade/transport) carry no demand of their own, so
    # total_demand(r,g)=0 and every use share above collapses to 0. The shares
    # would then sum to 0 and the margin flows (nm0/dm0), which build_Z spreads
    # through use_share_interm, would vanish from the table.
    #
    # Lacking any signal on who buys the margin good, we route it ENTIRELY to
    # intermediate use (0% to final demand) and split it across absorbing
    # sectors using the region's average intermediate-purchase profile.
    # --------------------------------------------------------------------------

    # Per sector s: its total intermediate demand across all goods g
    # (= column sum of the intermediate matrix). Shape (n, S).
    total_inputs = id0_df.sum(axis=1)

    # Region-wide total of intermediate purchases. Shape (n, 1).
    row_sum = total_inputs.sum(axis=1, keepdims=True)

    # "Representative buyer" profile: each sector's share of the region's total
    # intermediate purchases. Sums to 1 over s. Shape (n, S).
    fallback = total_inputs / np.where(row_sum < 1e-10, 1.0, row_sum)

    # Goods with zero total demand = the margin goods to patch. Shape (n, S, 1).
    mask_zero = (total_demand < 1e-10)[:, :, None]

    # For those goods only, overwrite the (over-s) interm shares with `fallback`
    # broadcast across the g axis. The interm shares now sum to 1 -> 100% of the
    # flow goes to intermediate demand; C/I/G stay 0 -> 0% to final demand.
    use_share_interm = np.where(mask_zero, fallback[:, np.newaxis, :], use_share_interm)

    return use_share_interm, use_share_C, use_share_I, use_share_G


In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# v1.2 margin routing — the two helpers consumed by build_Z_3 / build_F_3.
#   build_margin_tensors : load nm0 (r,g,m) and md0 (r,m,g) as dense tensors.
#   compute_use_shares_3 : DIRECT shares (as v2) + MARGIN shares routed via md0.
# ──────────────────────────────────────────────────────────────────────────────
def build_margin_tensors(year, regions, sectors, margins=('trd', 'trn')):
    """
    Build the margin tensors used to route trade/transport margins through Z and F.

    Returns
    -------
    nm0_rgm : (n, S, M)  national margin SUPPLIED by good g, of margin type m
    md0_rmg : (n, M, S)  margin DEMAND of type m in the absorption of good g
    m_set   : list of margin types m
    """
    n, S, M = len(regions), len(sectors), len(margins)
    margins = list(margins)

    nm0_rgm = (params['nm0_'][params['nm0_']['yr'] == year]
               .groupby(['r', 'g', 'm'])['value'].sum()
               .reindex(pd.MultiIndex.from_product([regions, sectors, margins],
                                                   names=['r', 'g', 'm']), fill_value=0.0)
               .values.reshape(n, S, M))

    md0_rmg = (params['md0_'][params['md0_']['yr'] == year]
               .groupby(['r', 'm', 'g'])['value'].sum()
               .reindex(pd.MultiIndex.from_product([regions, margins, sectors],
                                                   names=['r', 'm', 'g']), fill_value=0.0)
               .values.reshape(n, M, S))

    return nm0_rgm, md0_rmg, margins


def compute_use_shares_3(id0_df, cd0_mat, i0_mat, g0_mat, nm0_rgm, md0_rmg):
    """
    Use shares for the v1.2 build, split into a DIRECT and a MARGIN channel.
    Each channel sums to 1 over {buying sectors s} + {C, I, G} for every (r, g),
    so build_Z_3 / build_F_3 conserve dd0, dm0 and the bilateral T flows exactly.

    DIRECT  (d_int, dC, dI, dG): purchaser-price absorption structure of good g
        (id0/cd0/i0/g0 over total demand), with a fallback to the region's average
        input profile for goods that record no demand of their own.

    MARGIN  (m_int, mC, mI, mG): a unit of margin good g is a markup attached to the
        delivery of OTHER goods, so it is routed along the chain
            g --(nm0)--> margin type m --(md0)--> carrier good g' --(direct shares)--> buyer
        i.e. the margin lands on whoever buys the goods that carry it.
    """
    n, S = id0_df.shape[0], id0_df.shape[1]

    # DIRECT channel (identical to compute_use_shares_2)
    total_demand = id0_df.sum(axis=2) + cd0_mat + i0_mat + g0_mat
    safe = np.where(total_demand < 1e-10, 1.0, total_demand)
    d_int = id0_df / safe[:, :, None]
    dC, dI, dG = cd0_mat / safe, i0_mat / safe, g0_mat / safe

    # fallback: goods with no recorded demand (margin goods) -> avg input profile
    total_inputs = id0_df.sum(axis=1)
    row_sum = total_inputs.sum(axis=1, keepdims=True)
    fallback = total_inputs / np.where(row_sum < 1e-10, 1.0, row_sum)
    mask_zero = (total_demand < 1e-10)[:, :, None]
    d_int = np.where(mask_zero, fallback[:, None, :], d_int)

    # MARGIN channel (md0 chain)
    # margin-type mix supplied by good g (share over m). nm0-only == nm0+dm0 here.
    w = nm0_rgm.astype(float)
    w_sum = w.sum(axis=2, keepdims=True)
    w = np.divide(w, w_sum, out=np.zeros_like(w), where=w_sum > 1e-12)

    # buyer profile of each margin type m: who absorbs the goods that carry it
    tot_md = md0_rmg.sum(axis=2)                          # (n, M)
    safe_md = np.where(tot_md < 1e-12, 1.0, tot_md)
    bf_int = np.einsum('rmh,rhs->rms', md0_rmg, d_int) / safe_md[:, :, None]
    bf_C = (md0_rmg * dC[:, None, :]).sum(axis=2) / safe_md
    bf_I = (md0_rmg * dI[:, None, :]).sum(axis=2) / safe_md
    bf_G = (md0_rmg * dG[:, None, :]).sum(axis=2) / safe_md

    # weight the buyer profiles by good g's margin-type mix
    m_int = np.einsum('rgm,rms->rgs', w, bf_int)
    mC = np.einsum('rgm,rm->rg', w, bf_C)
    mI = np.einsum('rgm,rm->rg', w, bf_I)
    mG = np.einsum('rgm,rm->rg', w, bf_G)

    return d_int, dC, dI, dG, m_int, mC, mI, mG


In [ ]:
def build_Z(dd0_mat, nd0_mat, use_share_interm, T_all, sectors, n, S, xd0_mat=None):
    """
    xd0_mat : (n, S) supply to local market = dd0 + dm0.
              If None, falls back to dd0_mat (old behaviour, dm0 missing).
    Row sum  : xd0[r,g] + xn0[r,g] + x0[r,g]  = s0 (true WiNDC supply)
    """
    local_source = xd0_mat if xd0_mat is not None else dd0_mat

    Z_4d = np.zeros((n, S, n, S))

    for r_i in range(n):
        # xd0 = dd0 + dm0 : local margin revenue now included in row sums
        Z_4d[r_i, :, r_i, :] = local_source[r_i, :, None] * use_share_interm[r_i, :, :]

    for g_i, g in enumerate(sectors):
        T_g = T_all[g]
        ush = use_share_interm[:, g_i, :]
        Z_4d[:, g_i, :, :] += T_g[:, :, None] * ush[None, :, :]

    return Z_4d.reshape(n * S, n * S)


In [ ]:
def build_Z_3(dd0_mat, dm0_mat, T_dir, T_mar,
              ush_dir, ush_mar, sectors, n, S):
    Z = np.zeros((n, S, n, S))
    for r in range(n):                                  # local: dd0 direct + dm0 margin
        Z[r, :, r, :] += dd0_mat[r, :, None] * ush_dir[r, :, :]
        Z[r, :, r, :] += dm0_mat[r, :, None] * ush_mar[r, :, :]
    for gi, g in enumerate(sectors):                    # interstate flows
        Z[:, gi, :, :] += T_dir[g][:, :, None] * ush_dir[:, gi, :][None, :, :]
        Z[:, gi, :, :] += T_mar[g][:, :, None] * ush_mar[:, gi, :][None, :, :]
    return Z.reshape(n * S, n * S)


def build_F_3(dd0_mat, dm0_mat, T_dir, T_mar,
              dC, dI, dG, mC, mI, mG, sectors, n, S):
    F = np.zeros((n, S, n, 3))
    for r in range(n):
        F[r, :, r, 0] += dd0_mat[r, :] * dC[r, :] + dm0_mat[r, :] * mC[r, :]
        F[r, :, r, 1] += dd0_mat[r, :] * dI[r, :] + dm0_mat[r, :] * mI[r, :]
        F[r, :, r, 2] += dd0_mat[r, :] * dG[r, :] + dm0_mat[r, :] * mG[r, :]
    for gi, g in enumerate(sectors):
        F[:, gi, :, 0] += T_dir[g] * dC[:, gi][None, :] + T_mar[g] * mC[:, gi][None, :]
        F[:, gi, :, 1] += T_dir[g] * dI[:, gi][None, :] + T_mar[g] * mI[:, gi][None, :]
        F[:, gi, :, 2] += T_dir[g] * dG[:, gi][None, :] + T_mar[g] * mG[:, gi][None, :]
    return F.reshape(n * S, n * 3)


In [ ]:
def build_F(dd0_mat, T_all, use_share_C, use_share_I, use_share_G, sectors, n, S, xd0_mat=None):
    """
    Build final demand matrix F (n·S × n·3).
    Rows: (origin region, good). Columns: (destination region, {C, I, G}).
    """
    local_src = xd0_mat if xd0_mat is not None else dd0_mat  # ← only change
    
    F_4d = np.zeros((n, S, n, 3))

    for r_i in range(n):
        F_4d[r_i, :, r_i, 0] = local_src[r_i, :] * use_share_C[r_i, :]
        F_4d[r_i, :, r_i, 1] = local_src[r_i, :] * use_share_I[r_i, :]
        F_4d[r_i, :, r_i, 2] = local_src[r_i, :] * use_share_G[r_i, :]

    for g_i, g in enumerate(sectors):
        T_g = T_all[g]
        F_4d[:, g_i, :, 0] += T_g * use_share_C[:, g_i][None, :]  # fix: += not =
        F_4d[:, g_i, :, 1] += T_g * use_share_I[:, g_i][None, :]
        F_4d[:, g_i, :, 2] += T_g * use_share_G[:, g_i][None, :]

    return F_4d.reshape(n * S, n * 3)


def build_VA_EX(ld0_mat, kd0_mat, x0_mat, n, S):
    """Value added (labor + capital) and international exports, both flattened to n·S."""
    VA = (ld0_mat + kd0_mat).reshape(n * S)
    EX = x0_mat.reshape(n * S)
    return VA, EX


## Construction (both methods)

In [ ]:
YEAR = '2017'
names = ['dd0_', 'nd0_', 'xn0_', 'xd0_', 'x0_', 'm0_',
         'cd0_', 'i0_', 'g0_', 'ld0_', 'kd0_', 'ty0_']
data = load_year_data(YEAR, regions, sectors, names)
dd0_mat = data['dd0_']; nd0_mat = data['nd0_']
xn0_mat = data['xn0_']; xd0_mat = data['xd0_']; x0_mat = data['x0_']
m0_mat  = data['m0_'];  cd0_mat = data['cd0_']
i0_mat  = data['i0_'];  g0_mat  = data['g0_']
ld0_mat = data['ld0_']; kd0_mat = data['kd0_']
id0_df  = data['id0']
nm0_mat = (params['nm0_'][params['nm0_']['yr'] == YEAR]
           .groupby(['r', 'g'])['value'].sum().unstack('g')
           .reindex(index=regions, columns=sectors, fill_value=0.0).fillna(0.0).values)
print('data loaded for', YEAR)

In [ ]:
# Bilateral reconstructions used by the two methods (same export pool xn0)
T_new, _ = reconstruct_bilateral_2(
    xn0_mat, nd0_mat, nm0_mat, sectors, sector_to_idx, regions, D_np, gamma=1.0)
T3_dir, T3_mar, _ = reconstruct_bilateral_3(
    xn0_mat, nd0_mat, nm0_mat, sectors, sector_to_idx, regions, D_np,
    gamma_trade=1.0, gamma_margin=1.0)
print('reconstructed:', sum(T_new[g].sum() > 0 for g in sectors), '(v2) /',
      sum(T3_dir[g].sum() > 0 for g in sectors), '(v3) of', S)

In [ ]:
dm0_mat = xd0_mat - dd0_mat

# v2 -- aggregation: single use-share set, combined T_new, local source xd0
us_int, us_C, us_I, us_G = compute_use_shares_2(id0_df, cd0_mat, i0_mat, g0_mat)
Z_v2 = build_Z(dd0_mat, nd0_mat, us_int, T_new, sectors, n, S, xd0_mat=xd0_mat)
F_v2 = build_F(dd0_mat, T_new, us_C, us_I, us_G, sectors, n, S, xd0_mat=xd0_mat)

# v3 -- md0 chain: separate trade/margin flows, margin-specific use shares
nm0_rgm, md0_rmg, m_set = build_margin_tensors(YEAR, regions, sectors)
d_int, dC, dI, dG, m_int, mC, mI, mG = compute_use_shares_3(
    id0_df, cd0_mat, i0_mat, g0_mat, nm0_rgm, md0_rmg)
Z_v3 = build_Z_3(dd0_mat, dm0_mat, T3_dir, T3_mar, d_int, m_int, sectors, n, S)
F_v3 = build_F_3(dd0_mat, dm0_mat, T3_dir, T3_mar, dC, dI, dG, mC, mI, mG, sectors, n, S)

# reference -- margins EXCLUDED (local source dd0, no dm0)
Z_ref = build_Z(dd0_mat, nd0_mat, us_int, T_new, sectors, n, S)
F_ref = build_F(dd0_mat, T_new, us_C, us_I, us_G, sectors, n, S)

VA = (ld0_mat + kd0_mat).reshape(n * S)
EX = x0_mat.reshape(n * S)
print(f'v2: Z={Z_v2.sum():.1f} F={F_v2.sum():.1f} | v3: Z={Z_v3.sum():.1f} F={F_v3.sum():.1f}')

## Identity comparison

In [ ]:
# ── Identity summary (same columns as the old build_IOT comparison cell) ──────
Y = (params['ys0_'][params['ys0_']['yr'] == YEAR].groupby(['r', 's'])['value'].sum()
     .unstack('s').reindex(index=regions, columns=sectors, fill_value=0.0)
     .fillna(0.0).values.reshape(n * S))
ty0_mat  = load_matrix('ty0_', YEAR, regions, sectors)
tm0_mat  = load_matrix('tm0_', YEAR, regions, sectors)
ta0_mat  = load_matrix('ta0_', YEAR, regions, sectors)
rx0_mat  = load_matrix('rx0_', YEAR, regions, sectors)
# Full taxes (production + import tariffs + absorption), matching the saved v3 DB.
# Margins are NOT a separate field: the md0 chain routes them into Z and F.
tax_flat = ((ty0_mat * Y.reshape(n, S))
          + (tm0_mat * m0_mat)
          + (ta0_mat * (dd0_mat + nd0_mat + m0_mat - rx0_mat))).reshape(n * S)
M_imp    = (m0_mat[:, :, None] * d_int).sum(axis=1).reshape(n * S)   # ROW imports, direct shares
s0       = (xd0_mat + xn0_mat + x0_mat)                              # full WinDC supply

def _identity_row(name, Z, F):
    row     = (Z.sum(axis=1) + F.sum(axis=1) + EX).reshape(n, S)
    row_gap = s0 - row
    resid   = Y - (Z.sum(axis=0) + M_imp + VA + tax_flat)
    mask    = Y > 0.1
    return {'approach': name, 'sumZ': Z.sum(), 'sumF': F.sum(),
            'row gap % vs s0'  : row_gap.sum() / s0.sum() * 100,
            'row S|err|/Ss0 %' : abs(row_gap).sum() / s0.sum() * 100,
            'col resid %'      : resid.sum() / Y.sum() * 100,
            'col S|err|/SY %'  : abs(resid[mask]).sum() / Y[mask].sum() * 100}

summary = pd.DataFrame([
    _identity_row('margins excluded (ref)', Z_ref, F_ref),
    _identity_row('v2 aggregation',         Z_v2,  F_v2),
    _identity_row('v3 md0 chain',           Z_v3,  F_v3),
]).set_index('approach')
print(f'=== Z/F identity comparison ({YEAR}) -- sum s0 = {s0.sum():.1f} $bn ===')
print(summary.round(3).to_string())

## Why the column identity differs

In [ ]:
# ── Why v3's column identity is better: the margin Z->F reclassification ───────
colZ_v2, colZ_v3 = Z_v2.sum(), Z_v3.sum()
resid_v2 = (Y - (Z_v2.sum(axis=0) + M_imp + VA + tax_flat)).sum()
resid_v3 = (Y - (Z_v3.sum(axis=0) + M_imp + VA + tax_flat)).sum()
print(f'sum Z  : v2={colZ_v2:.1f}  v3={colZ_v3:.1f}  diff(v2-v3)={colZ_v2-colZ_v3:+.1f}')
print(f'col residual : v2={resid_v2:+.1f}  v3={resid_v3:+.1f}  diff(v3-v2)={resid_v3-resid_v2:+.1f}')
print()
print('The residual gap equals the Z gap: v2 routes ~{:.0f} $bn of margins that belong'.format(colZ_v2-colZ_v3))
print('to FINAL demand into intermediate use instead (margin goods have ~0 direct')
print('absorption, so v2 falls back to an average-input key; v3 sends them to the')
print('buyers of the goods that carry the margin via the md0 chain).')
# margin good wht: intermediate absorption under each method
gi = sector_to_idx['wht']
wht_v2 = Z_v2.reshape(n, S, n, S)[:, gi, :, :].sum()
wht_v3 = Z_v3.reshape(n, S, n, S)[:, gi, :, :].sum()
print(f"\nwht (wholesale) routed to intermediate:  v2={wht_v2:.1f}  v3={wht_v3:.1f}")

## Per-sector residual

In [ ]:
# ── Per-sector residuals: % of output + $ amounts, with output overlays — v2 vs v3 ──
# taxes = full (prod+import+absorption); margins are routed into Z/F (no separate field).
import matplotlib.pyplot as plt

s0_flat = s0.reshape(n * S)
out_sec   = Y.reshape(n, S).sum(axis=0)              # gross output per sector ($bn)
out_share = out_sec / out_sec.sum() * 100            # % of total output
out_safe  = np.where(out_sec > 0.1, out_sec, np.nan)

def _abs_by_sector(num_flat):                         # Sum_r|err| per sector ($bn)
    return np.abs(num_flat).reshape(n, S).sum(axis=0)

def r_col(Z, F): return Y       - (Z.sum(0) + M_imp + VA + tax_flat)                    # cost vs ys0
def r_row(Z, F): return s0_flat - (Z.sum(1) + F.sum(1) + EX)                            # supply closure
def r_rc (Z, F): return (Z.sum(1) + F.sum(1) + EX) - (Z.sum(0) + M_imp + VA + tax_flat) # sales vs production

models = {'v2': (Z_v2, F_v2), 'v3': (Z_v3, F_v3)}
colors = {'v2': '#d62728', 'v3': '#1f77b4'}
PANELS = [('Column identity', 'col', r_col),
          ('Row identity',    'row', r_row),
          ('Col = Row',       'rc',  r_rc)]

fig, axes = plt.subplots(3, 2, figsize=(20, 16))
rows = []
for (title, tag, fn), (axL, axR) in zip(PANELS, axes):
    resid_abs = {name: _abs_by_sector(fn(Z, F)) for name, (Z, F) in models.items()}          # $bn
    resid_pct = {name: np.where(out_sec > 0.1, resid_abs[name] / out_safe * 100, 0.0)
                 for name in models}                                                          # % of output
    order = np.argsort(out_sec)[::-1]                # sectors sorted by decreasing output

    x = np.arange(S)

    # LEFT: residual as % of OUTPUT (bars) + output-share overlay (secondary axis)
    for i, name in enumerate(models):
        axL.bar(x + (i - 0.5) * 0.4, resid_pct[name][order], width=0.4, color=colors[name], alpha=0.85, label=name)
    axL.axhline(5, color='orange', ls='--', lw=0.8)
    axL.set_xticks(x); axL.set_xticklabels([sectors[j] for j in order], rotation=90, fontsize=6)
    axL.set_ylabel('residual (% of output)'); axL.set_title(f'{title}  -  residual as % of output', fontsize=10)
    axL.legend(loc='upper right', fontsize=8)
    a2 = axL.twinx(); a2.plot(x, out_share[order], 'k.-', ms=4, lw=1.2, label='output share')
    a2.set_ylabel('output share (%)'); a2.set_ylim(0, out_share.max() * 1.15); a2.legend(loc='upper center', fontsize=8)

    # RIGHT: output & residual in $bn  (residual bars + output line on secondary axis)
    for i, name in enumerate(models):
        axR.bar(x + (i - 0.5) * 0.4, resid_abs[name][order], width=0.4, color=colors[name], alpha=0.85, label=f'{name} residual')
    axR.set_xticks(x); axR.set_xticklabels([sectors[j] for j in order], rotation=90, fontsize=6)
    axR.set_ylabel('residual ($bn)'); axR.set_title(f'{title}  -  output & residual ($bn)', fontsize=10)
    axR.legend(loc='upper right', fontsize=8)
    a3 = axR.twinx(); a3.plot(x, out_sec[order], 'k.-', ms=4, lw=1.2, label='output ($bn)')
    a3.set_ylabel('output ($bn)'); a3.legend(loc='upper center', fontsize=8)

    for name in models:
        v = resid_pct[name][out_sec > 0.1]
        rows.append({'identity': tag, 'model': name,
                     'resid $bn': round(resid_abs[name].sum(), 1),
                     'output $bn': round(out_sec.sum(), 1),
                     'resid %out (wtd)': round(resid_abs[name].sum() / out_sec.sum() * 100, 2),
                     'mean%': round(v.mean(), 2), 'median%': round(np.median(v), 2),
                     'p95%': round(np.percentile(v, 95), 2), 'max%': round(v.max(), 2)})
plt.tight_layout(); plt.show()

print('Per-sector residual summary ($ and % of output):')
print(pd.DataFrame(rows).to_string(index=False))

Conclusion : v3>>v2